# 05 · LoRA → mixed Q4 GGUF with imatrix (for local LM Studio)

**Part 1 (Unsloth):**
- The LoRA (`colab-ai/sft/<RUN_NAME>/lora`; `RUN_NAME` is set in the first cell of both parts) is merged into the base model
  with the same proven method as in `04`: `/content/merged`.

**Part 2 (llama.cpp):**
1. The merged model is converted to a bf16 GGUF.
2. An **imatrix** is computed on calibration text: the model's own math, physics, code and tool solutions, plus Turkish and English Wikipedia.
3. A **mixed Q4** is produced:

| Tensors | Type | Why |
|---|---|---|
| Experts (`ffn_*_exps`) | Q4_K_M | ~90% of the size; kept on the CPU locally |
| Attention, linear attention, shared expert | Q8_0 | Small but sensitive; on the GPU locally |
| Output layer / embedding | Q8_0 / Q6_K | |

Target size **~22–23 GB**, i.e. the same memory and speed as Unsloth's UD-Q4_K_XL.

The result is uploaded to a Hugging Face repo and downloaded into LM Studio from there.

**Runtime and cost:**
- Runtime: **A100 80GB** (the imatrix runs the whole bf16 model on the GPU).
- Estimate: ~1.5 hours, **~10 CU**.
- **Restart the session** between the two parts (to free GPU memory). **Do not** use "Disconnect and delete runtime": `/content` would be deleted.

# Part 1: Merge the LoRA

In [ ]:
# Same setup as 04 (versions are not pinned)
!pip install -q -U unsloth unsloth_zoo
!pip install -q "transformers==5.2.0"
!pip install -q -U nvidia-nccl-cu13

In [ ]:
import torch, torchvision.ops
print(torch.__version__, torch.version.cuda, torch.cuda.is_available(), torch.cuda.get_device_name(0))

In [ ]:
import os, shutil
from google.colab import drive, userdata
from huggingface_hub import login, snapshot_download

DRIVE_ROOT = "/content/drive/MyDrive/colab-ai"
RUN_NAME = "rft2"
LORA_DIR = f"{DRIVE_ROOT}/sft/{RUN_NAME}/lora"
MERGED_DIR = "/content/merged"

drive.mount("/content/drive")
login(token=userdata.get("HF_TOKEN"))
assert os.path.exists(f"{LORA_DIR}/adapter_config.json"), f"{LORA_DIR} not found: Part 1 of 04 must finish first"

In [ ]:
from unsloth import FastModel

# Loading from the LoRA folder downloads the base model (from adapter_config.json) and applies the adapter
model, processor = FastModel.from_pretrained(model_name=LORA_DIR, max_seq_length=4096,
                                             load_in_4bit=False, load_in_16bit=True)
tokenizer = getattr(processor, "tokenizer", processor)
model.save_pretrained_merged(MERGED_DIR, tokenizer, save_method="merged_16bit")

# Tokenizer + chat template from the original base model; missing helper files are copied too (same as 04)
base = snapshot_download("Qwen/Qwen3.6-35B-A3B-FP8", allow_patterns=["*.json", "*.jinja", "*.txt"])
TOKENIZER_FILES = {"tokenizer.json", "tokenizer_config.json", "vocab.json", "merges.txt", "chat_template.jinja",
                   "added_tokens.json", "special_tokens_map.json"}
for f in os.listdir(base):
    if f.endswith(".safetensors.index.json") or f in ("config.json", "generation_config.json"):
        continue
    if f in TOKENIZER_FILES or not os.path.exists(f"{MERGED_DIR}/{f}"):
        shutil.copy(f"{base}/{f}", MERGED_DIR)

# Disk: the downloaded bf16 base model (~67 GB) is no longer needed. Recent huggingface_hub versions keep files in a
# shared hub/blobs folder, so deleting the model folder alone frees nothing; the whole cache is deleted
# (everything needed was copied to MERGED_DIR).
shutil.rmtree(os.path.expanduser("~/.cache/huggingface/hub"), ignore_errors=True)
!du -sh {MERGED_DIR}; df -h /content | tail -1

# Part 2: GGUF

**Now:** *Runtime → Restart session*, then continue below.

In [ ]:
import os, sys, json, random, subprocess, glob, importlib, shutil
from google.colab import drive, userdata
from huggingface_hub import login, HfApi

DRIVE_ROOT = "/content/drive/MyDrive/colab-ai"
EVAL_DIR = f"{DRIVE_ROOT}/eval"
GEN_PATH = f"{DRIVE_ROOT}/rft/round1/generations.jsonl"
MERGED_DIR = "/content/merged"
RUN_NAME = "rft2"
NAME = f"Qwen3.6-35B-A3B-{RUN_NAME}"
# Separate NVMe disk on A100 machines (~350 GB free); otherwise /content (merged + bf16 GGUF ≈ 135 GB may not fit)
WORK = "/mnt/local-scratch" if os.path.isdir("/mnt/local-scratch") else "/content"
BF16 = f"{WORK}/{NAME}-BF16.gguf"
QUANT = f"{WORK}/{NAME}-imatrix-Q4_K_M.gguf"   # Q4_K_M base type with Q8_0 overrides (see below)
IMATRIX = f"{WORK}/{NAME}-imatrix.gguf"
CALIB = "/content/calibration.txt"
HF_REPO_NAME = f"{NAME}-GGUF"     # your username is prepended automatically
HF_PRIVATE = False
GITHUB_URL = "https://github.com/salihyesil59/qwen3.6-35b-a3b-rft"
SEED = 3407

drive.mount("/content/drive")
login(token=userdata.get("HF_TOKEN"))
assert os.path.exists(f"{MERGED_DIR}/config.json"), "No merged model: Part 1 must finish first"


def run(cmd):
    # Streams the output of long commands into the cell; stops on failure
    print("$", " ".join(cmd))
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError(f"Command failed (exit code {proc.returncode})")

In [ ]:
# llama.cpp (CUDA build for the A100 architecture only: ~10 min)
!git clone --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
!cmake -S /content/llama.cpp -B /content/llama.cpp/build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=80 -DLLAMA_CURL=OFF -DCMAKE_BUILD_TYPE=Release > /dev/null
!cmake --build /content/llama.cpp/build -j --target llama-quantize llama-imatrix llama-simple 2>&1 | tail -3
!pip install -q sentencepiece protobuf

In [ ]:
# HF → bf16 GGUF (~67 GB). The vision encoder is not included; the LoRA does not touch it, so for image
# support the mmproj file from Unsloth's repo can be used as is.
run([sys.executable, "/content/llama.cpp/convert_hf_to_gguf.py", MERGED_DIR, "--outtype", "bf16", "--outfile", BF16])
shutil.rmtree(MERGED_DIR)    # disk: no longer needed
!ls -lh {WORK}/*.gguf; df -h /content {WORK}

In [ ]:
# Tensor names: to see that the override patterns below actually match
sys.path.insert(0, "/content/llama.cpp/gguf-py")
import re, collections
from gguf import GGUFReader
reader = GGUFReader(BF16)
kinds = collections.Counter(re.sub(r"blk\.\d+\.", "blk.N.", t.name) for t in reader.tensors)
for k, n in sorted(kinds.items()):
    print(f"{n:>4} × {k}")
del reader

## Calibration text + imatrix

The calibration text consists of:
- **~300 RFT solutions** in the model's own thinking format, with the chat template applied. Math, physics, code,
  tool calls and multi-turn agent episodes are mixed.
- **80 articles each from Turkish and English Wikipedia** (the model is used in both languages).

This way quantization keeps the weights that matter most for the target domains more precise.

In [ ]:
from transformers import AutoTokenizer
from datasets import load_dataset

sys.path.insert(0, EVAL_DIR)
sys.path_importer_cache.pop(EVAL_DIR, None)
importlib.invalidate_caches()
import run_eval as R

tok = AutoTokenizer.from_pretrained("Qwen/Qwen3.6-35B-A3B-FP8")


def render(task, text):
    reasoning, _, answer = text.partition("</think>")
    msgs = R.build_messages(task)
    msg = {"role": "assistant", "content": answer.strip(), "reasoning_content": reasoning.strip()}
    if task["type"] == "tool":
        msg["content"] = answer.split("<tool_call>")[0].strip()
        msg["tool_calls"] = [{"type": "function", "function": {"name": c["name"], "arguments": c["arguments"]}}
                             for c in R.tool_calls_of({"content": answer})]
    return tok.apply_chat_template(msgs + [msg], tools=task.get("tools"), tokenize=False)


rng = random.Random(SEED)
solutions = []
with open(GEN_PATH, encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)
        good = [s for s in row["samples"] if s["ok"]]
        if good:
            solutions.append(render(row["task"], rng.choice(good)["text"])[:12000])
AGENT_DATA = f"{DRIVE_ROOT}/rft/agent1/agent_sft.jsonl"
if os.path.exists(AGENT_DATA):   # include multi-turn tool use (Python + sandboxed coding) in the calibration too
    for line in open(AGENT_DATA, encoding="utf-8"):
        ex = json.loads(line)
        solutions.append(tok.apply_chat_template(ex["messages"], tools=ex["tools"], tokenize=False)[:12000])
rng.shuffle(solutions)
parts = solutions[:300]
for lang in ["tr", "en"]:
    wiki = load_dataset("wikimedia/wikipedia", f"20231101.{lang}", split="train", streaming=True)
    parts += [r["text"][:4000] for _, r in zip(range(80), wiki.shuffle(seed=SEED, buffer_size=2000))]
rng.shuffle(parts)
with open(CALIB, "w", encoding="utf-8") as f:
    f.write("\n\n".join(parts))
print(f"{len(parts)} parts, {os.path.getsize(CALIB) / 1e6:.1f} MB")

In [ ]:
run(["/content/llama.cpp/build/bin/llama-imatrix", "-m", BF16, "-f", CALIB, "-o", IMATRIX,
     "-ngl", "99", "-c", "2048", "--chunks", "256"])

## Mixed Q4

The base type is Q4_K_M, which the experts keep. The patterns below upgrade the sensitive tensors to Q8_0.
`applying manual override` lines in the output show that the patterns matched.

In [ ]:
OVERRIDES = [
    r"attn_(q|k|v|qkv|output|gate)\.weight=q8_0",      # the 10 full-attention layers
    r"ssm_(in|out|ba|alpha)\.weight=q8_0",            # the 30 linear-attention layers
    r"ffn_(gate|up|down)_shexp\.weight=q8_0",         # shared expert (runs for every token)
]
cmd = ["/content/llama.cpp/build/bin/llama-quantize", "--imatrix", IMATRIX,
       "--token-embedding-type", "q6_K", "--output-tensor-type", "q8_0"]
for o in OVERRIDES:
    cmd += ["--tensor-type", o]
run(cmd + [BF16, QUANT, "Q4_K_M", str(os.cpu_count())])
print(f"Size: {os.path.getsize(QUANT) / 2**30:.1f} GiB")

In [ ]:
# Smoke test (raw completion, no chat template): should produce coherent text
run(["/content/llama.cpp/build/bin/llama-simple", "-m", QUANT, "-n", "80", "-ngl", "99",
     "The capital of France is Paris. Newton's second law"])

## Upload to Hugging Face

In [ ]:
api = HfApi()
repo_id = f"{api.whoami()['name']}/{HF_REPO_NAME}"
api.create_repo(repo_id, private=HF_PRIVATE, exist_ok=True)
card = f'''---
license: apache-2.0
base_model: Qwen/Qwen3.6-35B-A3B
tags: [gguf, qwen3.6, rft, imatrix]
---
# {NAME} (GGUF)

Qwen3.6-35B-A3B fine-tuned with rejection sampling fine-tuning (RFT): bf16 LoRA trained on the model's own verified,
shortest correct solutions ({RUN_NAME}). Training code, data recipe and eval results: {GITHUB_URL}

Quant: Q4_K_M with imatrix; attention / linear-attention / shared expert in Q8_0, embedding Q6_K, output Q8_0.
Recommended: thinking mode, temperature 0.6, top_p 0.95, top_k 20.
'''
api.upload_file(path_or_fileobj=card.encode(), path_in_repo="README.md", repo_id=repo_id)
api.upload_file(path_or_fileobj=IMATRIX, path_in_repo=os.path.basename(IMATRIX), repo_id=repo_id)
api.upload_file(path_or_fileobj=QUANT, path_in_repo=os.path.basename(QUANT), repo_id=repo_id)
print(f"Uploaded: https://huggingface.co/{repo_id}")

## Finish: stop spending CU

In [ ]:
from google.colab import runtime
runtime.unassign()

## Local use (LM Studio)

1. **Download** `...-imatrix-Q4_K_M.gguf` from the HF repo (in the browser or with `hf download`).
2. **Place** the file in LM Studio's models folder, e.g. `~/.lmstudio/models/local/Qwen3.6-35B-A3B-rft2/`.
   For image support, put the `mmproj` file from Unsloth's repo in the same folder.
3. **Load** it with the same settings as UD-Q4_K_XL: MoE expert offload to CPU, 32k context, KV cache Q8_0.
4. **Check:**
   ```
   python eval/run_eval.py --tasks eval/tasks/personal.jsonl --label rft2-local
   ```
   Also check that tool calls work in Cline and that thinking is shown separately.